# SFT on HumanEval (Qwen2.5-Coder-3B, LoRA, T4)

Supervised fine-tuning of **Qwen2.5-Coder-3B-Instruct** on the HumanEval subset of `final_dataset_v2.csv` using **QLoRA** (4-bit + LoRA). Designed for Google Colab with **T4 GPU**.

Outputs LoRA adapters in PEFT format and a FedAvg-ready `lora_state_dict.pt`.

In [3]:
print('x')

x


In [2]:
# Check GPU (optional; skip if no NVIDIA GPU)
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as e:
    print("nvidia-smi not available:", e)

Mon Mar 16 09:59:00 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A4000               Off | 00000000:55:00.0 Off |                  Off |
| 41%   35C    P8              13W / 140W |     77MiB / 16376MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
!pip install --no-cache-dir "transformers>=4.36" "peft>=0.7" "trl>=0.7,<0.20" "datasets" "accelerate" "pandas" "torch>=2.4" "torchvision" "torchaudio"

Defaulting to user installation because normal site-packages is not writeable


## Data loading

In [5]:
import os
import pandas as pd

# Local path: same directory as this notebook (or set CSV_PATH / NOTEBOOK_DIR if needed)
try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()
CSV_PATH = os.path.join(NOTEBOOK_DIR, "human_eval_sft_ready.csv")
print("Using CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))

Using CSV: /home/jovyan/FED/HumanEval/human_eval_sft_ready.csv
Total rows: 164


In [6]:
df_he = df[df["dataset"] == "humaneval"].copy()
df_he = df_he.dropna(subset=["prompt", "canonical_solution"])
df_he["canonical_solution"] = df_he["canonical_solution"].astype(str).str.strip()
df_he = df_he[df_he["canonical_solution"].str.len() > 0]
assert len(df_he) == 164, f"Expected 164 HumanEval rows, got {len(df_he)}"
print("HumanEval rows:", len(df_he))

HumanEval rows: 164


## Parse prompt and build chat messages

In [7]:
SEP = "\n\nUser: "

def row_to_messages(row):
    prompt_str = str(row["prompt"]).strip()
    idx = prompt_str.find(SEP)
    if idx == -1:
        import warnings
        warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
        system_content = "You are an expert Python developer. Complete the function provided by the user."
        user_content = prompt_str.replace("System: ", "", 1).strip()
    else:
        system_content = prompt_str[:idx].replace("System: ", "", 1).strip()
        user_content = prompt_str[idx + len(SEP):].strip()
    solution = str(row["canonical_solution"]).strip()
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": solution},
    ]

messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
print("Built", len(messages_list), "message lists.")

Built 164 message lists.


/tmp/ipykernel_1159/1770581936.py:8: UserWarning: Row HumanEval/0: no '\n\nUser: ' found; using whole prompt as user.
  warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
/tmp/ipykernel_1159/1770581936.py:8: UserWarning: Row HumanEval/1: no '\n\nUser: ' found; using whole prompt as user.
  warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
/tmp/ipykernel_1159/1770581936.py:8: UserWarning: Row HumanEval/2: no '\n\nUser: ' found; using whole prompt as user.
  warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
/tmp/ipykernel_1159/1770581936.py:8: UserWarning: Row HumanEval/3: no '\n\nUser: ' found; using whole prompt as user.
  warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
/tmp/ipykernel_1159/1770581936.py:8: UserWarning: Row HumanEval/4: no '\n\nUser: ' found; using whole prompt as user.
  warnings.war

In [8]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"messages": messages_list})
print(train_dataset)

Dataset({
    features: ['messages'],
    num_rows: 164
})


In [9]:
print(len(train_dataset))

164


## Model and tokenizer

In [10]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token 
print("Tokenizer loaded.")

Tokenizer loaded.


In [11]:
import torch
from transformers import AutoModelForCausalLM

compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded (fp16/bf16, no quantization).")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded (fp16/bf16, no quantization).


## LoRA (PEFT) configuration

**FedAvg**: Use this exact config on all clients so state dict keys and shapes match when averaging.

In [13]:
from peft import LoraConfig, get_peft_model

LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

/opt/conda/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Training

In [14]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "./sft_humaneval_output/lora_adapters"

# Completion-only loss: mask prompt tokens (try DataCollatorForCompletionOnlyLM if available)
try:
    from trl import DataCollatorForCompletionOnlyLM
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
except ImportError:
    try:
        from trl.extras import DataCollatorForCompletionOnlyLM
        response_template = "<|im_start|>assistant\n"
        collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
    except ImportError:
        collator = None

training_args = SFTConfig(
    output_dir="./sft_humaneval_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    save_total_limit=1,
    max_seq_length=1024,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)
if collator is not None:
    trainer_kwargs["data_collator"] = collator

trainer = SFTTrainer(**trainer_kwargs)
print("SFTTrainer created.")
# print(len(train_datsets))

Applying formatting function to train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

SFTTrainer created.


In [16]:
# Optional: quick sanity run (comment out after verifying)
# trainer.train(max_steps=2)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.156740
10,0.077598
15,0.086312
20,0.066605
25,0.060938
30,0.041546
35,0.032581
40,0.046690
45,0.041032
50,0.032048


TrainOutput(global_step=63, training_loss=0.05678772021617208, metrics={'train_runtime': 180.7859, 'train_samples_per_second': 2.721, 'train_steps_per_second': 0.348, 'total_flos': 3261768043585536.0, 'train_loss': 0.05678772021617208})

## Save adapter (PEFT format)

In [18]:
import os
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter and tokenizer to", ADAPTER_DIR)
print("Files:", os.listdir(ADAPTER_DIR))

Saved adapter and tokenizer to ./sft_humaneval_output/lora_adapters
Files: ['adapter_model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'README.md', 'adapter_config.json']


## Get and print LoRA adapters (FedAvg-ready, visible as output)

In [19]:
lora_state = {
    k: v.detach().cpu().clone()
    for k, v in model.named_parameters()
    if v.requires_grad
}

print("=== LoRA adapter structure (name -> shape) ===")
for name, tensor in lora_state.items():
    print(f"  {name}: {tensor.shape}")

print("\n=== Per-parameter summary (min, max, norm) ===")
for name, tensor in lora_state.items():
    t = tensor.float()
    print(f"  {name}: min={t.min().item():.4f}, max={t.max().item():.4f}, norm={t.norm().item():.4f}")

total_params = sum(p.numel() for p in lora_state.values())
print(f"\n=== Summary ===")
print(f"  Number of LoRA parameters: {len(lora_state)}")
print(f"  Total elements: {total_params}")
print("  Parameter names (for FedAvg):", list(lora_state.keys()))

=== LoRA adapter structure (name -> shape) ===
  base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.base_model.model.model.l

**FedAvg**: Use `lora_state_dict.pt` below for aggregation. Load it with `torch.load(...)` and average the tensors with other clients' LoRA state dicts (same keys and shapes).

In [20]:
lora_pt_path = os.path.join(ADAPTER_DIR, "lora_state_dict.pt")
torch.save(lora_state, lora_pt_path)
print("Saved FedAvg-ready LoRA state dict to", lora_pt_path)
print("File size (MB):", os.path.getsize(lora_pt_path) / (1024 * 1024))

# Optional: copy to Google Drive to keep after session
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r ./sft_humaneval_output /content/drive/MyDrive/

Saved FedAvg-ready LoRA state dict to ./sft_humaneval_output/lora_adapters/lora_state_dict.pt
File size (MB): 114.36724758148193
